# Scene Classification — Colab walkthrough

Runs the full pipeline end-to-end in Google Colab: downloads MIT Indoor67, extracts YOLOv8 object features, trains 6 tabular classifiers and a transfer-learned ResNet50, and prints a comparison table.

**Tip.** Runtime → Change runtime type → T4 GPU (free tier) makes the CNN step ~10× faster.

Source: [github.com/simonyos/scene-classification](https://github.com/simonyos/scene-classification)

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
!git clone --depth 1 https://github.com/simonyos/scene-classification.git
%cd scene-classification
!pip install -q -e . --no-deps
!pip install -q numpy pandas scikit-learn shap ultralytics pillow matplotlib mlflow typer rich fastapi 'uvicorn[standard]' python-multipart pydantic tqdm tabulate

## 1. Download MIT Indoor67 and build splits

~2.4 GB tar, ~2 min on a Colab VM. By default subsets to the paper's 4 classes (bedroom, bathroom, kitchen, livingroom).

In [ ]:
!scenes prepare-data

## 2. Extract YOLOv8 object features

80 COCO classes × {count, confsum, areafrac} = 240 features per image.

In [ ]:
!scenes extract-features

## 3. Train tabular classifiers (DT, KNN, NB, SVM, LR, GradBoost)

In [ ]:
!scenes train-tabular

## 4. Train ResNet50 (transfer learning)

On a T4 GPU this takes ~5 minutes (15 epochs, early stopping on val accuracy).

In [ ]:
!scenes train-cnn

## 5. Evaluate: confusion matrices, comparison chart, summary

In [ ]:
!scenes evaluate
from IPython.display import Image, Markdown, display
display(Markdown(open('reports/summary.md').read()))
display(Image('reports/figures/comparison_bar.png'))
for f in sorted((p for p in __import__('pathlib').Path('reports/figures').iterdir() if p.name.startswith('confusion_'))):
    display(Markdown(f'### {f.stem}'))
    display(Image(str(f)))

## 6. Try the FastAPI endpoint on a sample image

Starts the server in the background, posts a random test image, and shows the prediction + detected objects.

In [ ]:
import subprocess, time, requests, random
from pathlib import Path

proc = subprocess.Popen(['uvicorn', 'scene_classification.serve.api:app', '--host', '0.0.0.0', '--port', '8000'])
time.sleep(6)

test_root = Path('data/processed/splits/test')
image_path = random.choice(list(test_root.rglob('*.jpg')))
print('Image:', image_path, '(true label:', image_path.parent.name, ')')

with open(image_path, 'rb') as f:
    r = requests.post('http://localhost:8000/predict', files={'image': f})
print(r.json())

proc.terminate()